In [12]:
#!pip install llm_feature_gen

In [13]:
"""
Quickstart: reach the university LLM endpoint (llmlite.vse.cz) with llm-feature-gen
and run a first feature-extraction pass on synthetic data.

Run locally:  pip install llm-feature-gen pandas
              python 00_llmlite_quickstart.py
"""
import json
import pandas as pd
from llm_feature_gen.providers.local_provider import LocalProvider

In [14]:
import os
from dotenv import load_dotenv
load_dotenv()  # loads LOCAL_OPENAI_API_KEY from the project-root .env (never hardcode the key)

# --- 1. Point the provider at the university endpoint ---------------------
BASE_URL = "https://litellm.vse.cz/v1"       # OpenAI-compatible LiteLLM proxy
API_KEY = os.environ["LOCAL_OPENAI_API_KEY"]        # from Tomas
MODEL = "qwen3.5:122b"                        # PLACEHOLDER - see step 2 below

In [15]:
provider = LocalProvider(
    base_url=BASE_URL,
    api_key=API_KEY,
    default_text_model=MODEL,
    temperature=0.0,
    max_tokens=1024,
)

In [16]:
# --- 2. First: list available models to get the REAL model name ----------
# (LocalProvider doesn't expose this directly, so we hit the endpoint's
#  OpenAI client directly - it's the same client under the hood.)
try:
    models = provider.client.models.list()
    print("Available models on llmlite.vse.cz:")
    for m in models.data:
        print(" -", m.id)
except Exception as e:
    print(f"Could not list models ({type(e).__name__}: {e}). "
          f"Falling back to MODEL='{MODEL}' as a guess.")

Available models on llmlite.vse.cz:
 - cellsense-fim-7b
 - qwen3-embedding:4b
 - gemma3:270m
 - qwen3.5:122b
 - qwen3.6-35b
 - nomic-embed-text-v2-moe:latest
 - ornith:35b
 - Qwen3.6-35B-A3B
 - Qwen3-Coder-Next:latest
 - ornith:9b
 - qwen3-vl:235b-a22b-instruct
 - qwen3-vl:32b-instruct
 - qwen3-vl:32b
 - ThinkingCap-Qwen3.6-27B-GGUF:Q8_0
 - google/medgemma-1.5-4b-it
 - medgemma-27b-it:q8


In [17]:
# --- 3. Synthetic data you can use right away in pandas -------------------
df = pd.DataFrame({
    "file": ["syn_001.txt", "syn_002.txt", "syn_003.txt"],
    "text": [
        "Vidím psa, jak běží po zahradě za míčem. Je tam taky strom a plot.",
        "Tady je nějaká... nevim, snad pes? A tam něco, co tady, jako.",
        "Na obrázku je dítě, které si hraje s míčem vedle domu a stromu.",
    ],
})

In [18]:
# --- 4. A simple label-blind extraction prompt -----------------------------
PROMPT = (
    "You will read a short description of a picture, written by a speaker. "
    "Extract these features as JSON: "
    "n_entities_named (integer, count of distinct concrete objects/people/animals named), "
    "hedging_present (yes/no, whether the speaker uses uncertain phrases like 'maybe', "
    "'I don't know', 'something'), "
    "sentence_style (one of: complete, fragmented, mixed). "
    "Return ONLY the JSON object, no explanation."
)

In [19]:
import socket

try:
    print(socket.gethostbyname("llmlite.vse.cz"))
except Exception as e:
    print("DNS resolution failed:", e)

DNS resolution failed: [Errno -2] Name or service not known


In [20]:
import os
from dotenv import load_dotenv
load_dotenv()  # loads LOCAL_OPENAI_API_KEY from the project-root .env (never hardcode the key)

from openai import OpenAI

client = OpenAI(
    base_url="https://litellm.vse.cz/v1",
    api_key=os.environ["LOCAL_OPENAI_API_KEY"]
)

models = client.models.list()

for model in models.data:
    print(model.id)

cellsense-fim-7b
qwen3-embedding:4b
gemma3:270m
qwen3.5:122b
qwen3.6-35b
nomic-embed-text-v2-moe:latest
ornith:35b
Qwen3.6-35B-A3B
Qwen3-Coder-Next:latest
ornith:9b
qwen3-vl:235b-a22b-instruct
qwen3-vl:32b-instruct
qwen3-vl:32b
ThinkingCap-Qwen3.6-27B-GGUF:Q8_0
google/medgemma-1.5-4b-it
medgemma-27b-it:q8


In [21]:
import requests

try:
    r = requests.get("https://llmlite.vse.cz", timeout=10)
    print(r.status_code)
    print(r.text[:500])
except Exception as e:
    print(type(e).__name__, e)

ConnectionError HTTPSConnectionPool(host='llmlite.vse.cz', port=443): Max retries exceeded with url: / (Caused by NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7857c6939f90>: Failed to resolve 'llmlite.vse.cz' ([Errno -2] Name or service not known)"))


In [22]:
# --- 5. Run it and put the results straight back into the DataFrame -------
results = provider.text_features(df["text"].tolist(), prompt=PROMPT, feature_gen=True)
feat_df = pd.DataFrame(results)
df = pd.concat([df.reset_index(drop=True), feat_df], axis=1)

ValueError: Invalid JSON response: 

In [ ]:
print("\nResult:")
print(df)
df.to_csv("llmlite_quickstart_results.csv", index=False)
print("\nSaved to llmlite_quickstart_results.csv")